In [1]:
import os
import random
import pandas as pd


# Mount to google drive
# from google.colab import drive
# drive.mount('/content/drive')

# Change it to your google drive path where this notebook located.
drive_path = '/Users/samyiin/Projects/ZipfLawAnalysis'
os.chdir(drive_path)

# Description
In this notebook we will sum up all the temp results we got. For a reminder, we get all the users 

    "type": 'user', "language": 'Python', 'created': '<2024-04-01', 'repos':">=1", "followers": ">=1"
Now we will do the following things:
1. Filter out those without email address. (Reasons below)
2. get some statistics for the next step: see how should we handle the location and other stuff. 
3. merge this with those 500+ USA programmers that we already sent email to, so that we will not send them emails again. 

    

In [2]:

# Specify the directory containing the CSV files
directory = "Database/TempData/GatherData/Filter_1_Parallel/TempResults/"

# List to hold individual DataFrames
dataframes = []

# Loop through all files in the directory
for filename in os.listdir(directory):
    if filename.endswith(".csv"):
        # Load each CSV file and append to the list
        file_path = os.path.join(directory, filename)
        df = pd.read_csv(file_path)
        dataframes.append(df)

# Concatenate all DataFrames in the list
df_users = pd.concat(dataframes, ignore_index=True)

# drop duplicates because there might be some 
df_users = df_users.drop_duplicates(subset=['id']).reset_index(drop=True)


In [3]:
df_users.columns

Index(['login', 'id', 'node_id', 'avatar_url', 'url', 'html_url',
       'followers_url', 'following_url', 'gists_url', 'starred_url',
       'subscriptions_url', 'organizations_url', 'repos_url', 'events_url',
       'received_events_url', 'type', 'user_view_type', 'site_admin', 'name',
       'company', 'blog', 'location', 'email', 'hireable', 'bio',
       'twitter_username', 'public_repos', 'public_gists', 'followers',
       'following', 'created_at', 'updated_at', 'gravatar_id', 'score'],
      dtype='object')

In [ ]:
# get rid off some url columns and useless columns. (Useless for our analysis) 
df_users = df_users[['login', 'id', 'type', 'site_admin', 'name',
       'company', 'blog', 'location', 'email', 'hireable', 'bio',
       'twitter_username', 'public_repos', 'public_gists', 'followers',
       'following', 'created_at', 'updated_at']]

## Email
We will select only those who have email, becasue later we will check if they are native through sending them emails. So those without email, although might be actual American/Chinese native programmers, since we cannot verify it, we will discard them. 

In [5]:
# select those who have an email: verifiable
df_users = df_users[~df_users['email'].isna()]

## Duplicates

In [ ]:
# how many accounts have the same email, for each email how many times is it duplicated?
df_users[df_users['email'].duplicated(keep=False)]['email'].value_counts()

In [ ]:
df_users[df_users['name'].duplicated(keep=False)]['name'].value_counts()

In [ ]:
df_users[df_users['twitter_username'].duplicated(keep=False)]['twitter_username'].value_counts()

In [ ]:
df_users[df_users['blog'].duplicated(keep=False)]['blog'].value_counts()

In [ ]:
df_users[df_users['bio'].duplicated(keep=False)]['bio'].value_counts()

In [ ]:
df_users[df_users['company'].duplicated(keep=False)]['company'].value_counts()

## Location

In [12]:
# how many unique location strings are there
df_users['location'].nunique()

30285

In [13]:
# how many people have location
df_users['location'].notna().sum()

np.int64(146620)

In [14]:
# how many people doesn't have location
df_users['location'].isna().sum()

np.int64(68708)

# Conclusion
If two account have the same email, we probably assume that they both belong to the same person. But duplicated names are much more common, and it's obvious that we cannot use it to conclude that both account belongs to the same person. Other good indicators are bio, company, blog and twitter_user_name. But either way, we see that there are negligible amount of people with duplicate accounts.

Regarding Location, determining the country is a complex task: There are easy-to-identify locations like "Beijing, China", and harder ones like "CZ, Nový Jičín". But we see that there are only around 30k different strings for locations, so using LLM might be a good way to identify the country of a person. 

In [ ]:
df_users

In [17]:
df_users.to_csv(f'Database/TempData/GatherData/Filter_1_Parallel/Results/users_filter_1.csv', index=False)